# Sanity — is the cluster actually serving, and through the gateway?

Run this **before any load test**. It answers six questions in order, and
each one makes the next meaningful:

1. what answers at all
2. what model the fleet is serving
3. how much KV each pod got, and what concurrency that buys
4. does a request really traverse the gateway
5. is the KV hop moving bytes or recording a row
6. was anything scarce

It writes `../reports/sanity-<date>.pdf`.

**Through the tunnel** (`make ssh`) from a laptop, or straight on the box.
Latency here is indicative only — the tunnel adds your round-trip. Use
`make load` for numbers you intend to quote.


In [ ]:
from datetime import datetime, timezone
from utils import Cluster, Report, sum_by_name, REPORTS

cluster = Cluster()   # 127.0.0.1:8080 gateway, :30800 prefill, :30801 decode
report = Report('Dodona — cluster sanity', 'one pass, no load')
print('reports go to', REPORTS)


## 1. What answers

Everything below is meaningless if the gateway is down, and a `False` on
`decode /metrics` means the pool never came up rather than that it is idle.


In [ ]:
report.heading('1. Endpoints')
up = cluster.endpoints()
for name, ok in up.items():
    report.verdict(ok, name)
    print(('ok   ' if ok else 'DOWN '), name)

if not up.get('gateway /metrics'):
    print()
    print('The gateway is not reachable. Is the tunnel open (make ssh)?')
    print('Is orch-serve running?  sudo k3s kubectl get pods')


## 2. What is it serving

The model id the gateway advertises has to match what the pods were started
with, or vLLM rejects every request with an unknown model.


In [ ]:
report.heading('2. Model and pools')
model = cluster.served_model()
report.verdict(bool(model), f'gateway advertises a model: {model}')
print('model:', model)

rows = []
for pool in ('prefill', 'decode'):
    m = cluster.pool_metrics(pool)
    rows.append([pool,
                 int(sum_by_name(m, 'vllm:num_requests_running')),
                 int(sum_by_name(m, 'vllm:num_requests_waiting')),
                 int(sum_by_name(m, 'vllm:prompt_tokens_total')),
                 int(sum_by_name(m, 'vllm:generation_tokens_total'))])
report.table(['pool', 'running', 'waiting', 'prompt tok', 'gen tok'], rows,
             note='prompt_tokens_total of 0 on a pool means nothing was ever routed to it')
for r in rows: print(r)


## 3. Capacity — what each pod actually got

`kv_cache_usage_perc` is a **ratio of that pod's own KV**, so it says nothing
about absolute size. The absolute number is in the pod's startup log:

```bash
make logs | grep -E 'Available KV cache memory|Maximum concurrency'
```

Fill the two values in below from that line. They are the denominator of
every concurrency claim in the submission, and guessing them is how a
whole-card figure ends up quoted for a slice.


In [ ]:
# From the pod log. Replace with what YOUR pods report.
KV_GIB_PER_POD = 4.3        # 'Available KV cache memory: X GiB'
SEQ_PER_POD    = 4.9        # 'Maximum concurrency for 16,384 tokens: Xx'
DECODE_REPLICAS = 1         # KEDA scales this to 2 under load

report.heading('3. Capacity')
pool_total = SEQ_PER_POD * DECODE_REPLICAS
report.table(['', 'value'], [
    ['KV per decode pod', f'{KV_GIB_PER_POD:.2f} GiB'],
    ['full-length seqs per pod', f'{SEQ_PER_POD:.1f}'],
    ['decode replicas', DECODE_REPLICAS],
    ['decode pool concurrency', f'{pool_total:.1f}'],
], note='A pool this small is the point: a policy needs something to be scarce.')
print('decode pool concurrency:', round(pool_total, 1))


## 4. Does a request traverse the gateway

This is the one that decides whether any of the control plane is real.
`orch_requests_total` only moves if the request went through our gateway —
a completion that reached a pod directly leaves it at zero, and from outside
the two look identical.


In [ ]:
report.heading('4. Does a request traverse the gateway')
before = sum_by_name(cluster.gateway_metrics(), 'orch_requests_total')
answer = cluster.chat('Reply with the single word: ok', max_tokens=8)
after  = sum_by_name(cluster.gateway_metrics(), 'orch_requests_total')

traversed = after > before
report.verdict(traversed, f'orch_requests_total {before:.0f} -> {after:.0f}')
report.verdict('_error' not in answer, f"completion returned in {answer['_latency_s']}s")

print('orch_requests_total', before, '->', after)
print('latency', answer.get('_latency_s'), 's')
if '_error' in answer:
    print('ERROR', answer['_error'], answer.get('_body', '')[:300])
else:
    ch = (answer.get('choices') or [{}])[0]
    print('content:', (ch.get('message') or {}).get('content'))
    print('via:', answer.get('via'), ' kv_hop:', answer.get('kv_hop'))


## 5. Is the KV hop real

`kv_hop` in the response body proves the router **recorded** a hop. It does
not prove bytes moved — that needs the connector, and the connector reports
its transport in the pod log:

```bash
make logs | grep -E 'HCAs|transport|Mooncake'
```

| what the log says | what it means |
|---|---|
| `Found 0 HCAs` · `Using TCP transport` | works, over loopback TCP, not RDMA |
| `Found 1 HCAs` · RDMA transport | the fast path |
| no Mooncake lines at all | no connector: decode re-prefills, the hop is a row |


In [ ]:
report.heading('5. The KV hop')
hop = answer.get('kv_hop') if isinstance(answer, dict) else None
report.verdict(bool(hop), f'router recorded a hop: {hop}')

# Fill in from the pod log — the notebook cannot read it through the tunnel.
MOONCAKE_TRANSPORT = 'unknown'   # 'rdma' | 'tcp' | 'none'
report.verdict(MOONCAKE_TRANSPORT in ('rdma', 'tcp'),
               f'connector transport: {MOONCAKE_TRANSPORT}')
if MOONCAKE_TRANSPORT == 'tcp':
    report.text('TCP means /dev/infiniband is not visible in the pod. Blocks still')
    report.text('move, but not at RDMA speed.')
print('kv_hop:', hop)
print('transport:', MOONCAKE_TRANSPORT)


### Does decode re-prefill?

A cheap, external signal. If the connector moves blocks, decode should NOT
have to compute the prompt — so `prefix_cache_hits_total / queries_total` on
the decode pool rises, and its time-to-first-token on a long prompt collapses.

`prompt_tokens_total` being equal across pools is suggestive but not proof:
vLLM counts the tokens in the request whether or not they were cached. The
prefix-cache ratio is the honest one.


In [ ]:
rows = []
for pool in ('prefill', 'decode'):
    m = cluster.pool_metrics(pool)
    hits = sum_by_name(m, 'vllm:prefix_cache_hits_total')
    qs   = sum_by_name(m, 'vllm:prefix_cache_queries_total')
    rows.append([pool, int(sum_by_name(m, 'vllm:prompt_tokens_total')),
                 int(hits), int(qs), f'{hits / qs:.1%}' if qs else 'n/a'])
report.table(['pool', 'prompt tok', 'prefix hits', 'queries', 'hit rate'], rows,
             note='Decode loading blocks instead of recomputing shows up as a high hit rate here.')
for r in rows: print(r)

pre, dec = rows[0][1], rows[1][1]
if pre and dec and abs(pre - dec) < max(pre, dec) * 0.05:
    report.text(f'Both pools report ~{pre:,} prompt tokens. Consistent with decode')
    report.text('recomputing the prompt rather than loading it. Confirm with TTFT.')
    print()
    print(f'NOTE both pools at ~{pre:,} prompt tokens — decode may be re-prefilling')


## 6. Was anything scarce

The question that separates a result from a number. If KV never rises, an
admission policy reading it is **inert** — installed, running on every
request, unable to change any outcome.


In [ ]:
report.heading('6. Was anything scarce')
kv = cluster.kv_usage()
for pool, v in kv.items():
    print(f'{pool:9} kv_cache_usage_perc {v:.4f}')
report.table(['pool', 'kv_cache_usage_perc'],
             [[p, f'{v:.4f}'] for p, v in kv.items()],
             note='Idle cluster: near zero is expected here. make load is what moves it.')
report.text('This is an idle reading. Scarcity is a load-test question;')
report.text('see make load and the run report.')

def kv_chart(ax):
    pools = list(kv) or ['none']
    ax.bar(pools, [kv.get(p, 0) for p in pools])
    ax.axhline(0.90, ls='--', color='#c0392b')
    ax.set_ylim(0, 1); ax.set_ylabel('kv_cache_usage_perc')
    ax.set_title('KV usage vs the 0.90 admission threshold')
report.chart(kv_chart, note='Dashed line is the kv_threshold policy trigger.')


## Save the report


In [ ]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
path = report.save(f'sanity-{stamp}')
fails = report.failures()
print('wrote', path)
print()
if fails:
    print('FAILURES — fix these before running load:')
    for f in fails: print('  -', f)
else:
    print('All checks passed. Next:  make load')
